# Decision Support System (DSS) - M.Sc. in Computer Science
## Context-Aware Music Recommendation & Health-Routine Optimization

**Student:** Stefano  
**Course:** Decision Support Systems / Knowledge-Based Systems  

---

### Project Abstract
Conventional music recommendation platforms (such as Spotify, Apple Music, and YouTube Music) suffer from the **"context-blind recommendation"** problem: they propose tracks based solely on historical listening logs or aggregate popularity, ignoring **how the listener actually feels** in real time.

This **Decision Support System (DSS)** bridges this gap by fusing:
1. **Spotify Mathematical Audio Features** (BPM tempo, affective valence, energy intensity, instrumental purity).
2. **Smartwatch Real-Time Biometric Telemetry** (heart rate BPM, stress index on a 0-100 scale, current physical/cognitive activity).

Leveraging a **hierarchical AND-OR inference tree (meta-rules, domain sub-rules, and negative medical safety barriers)** in Prolog, the DSS evaluates physiological feasibility and prescribes the optimal track or a staged entrainment session (*Altshuler's Iso-Principle*), providing transparent biofeedback explanations.

## 1. Dataset Loading and Exploratory Inspection
The system ingests raw structured JSON catalogs stored in the `data/` directory.

In [1]:
import json
from pathlib import Path

with open('data/spotify_tracks.json', 'r', encoding='utf-8') as f:
    tracks = json.load(f)
with open('data/smartwatch_biometrics.json', 'r', encoding='utf-8') as f:
    users = json.load(f)

print(f"[+] Loaded {len(tracks)} Spotify tracks and {len(users)} smartwatch biometric profiles successfully.")
print("\n--- Sample Smartwatch User Telemetry ---")
for u in users[:3]:
    print(f" * {u['name']} ({u['user_id']}) -> Activity: {u['current_activity']} | Heart Rate: {u['heart_rate_bpm']} BPM | Stress: {u['stress_level']}/100")


[+] Loaded 30 Spotify tracks and 6 smartwatch biometric profiles successfully.

--- Sample Smartwatch User Telemetry ---
 * Sarah Miller (user_sarah) -> Activity: high_stress_work | Heart Rate: 114.0 BPM | Stress: 84.0/100
 * Alex Chen (user_alex) -> Activity: cardio_running | Heart Rate: 146.0 BPM | Stress: 45.0/100
 * Marco Rossi (user_marco) -> Activity: deep_focus_study | Heart Rate: 74.0 BPM | Stress: 32.0/100


## 2. Automated Knowledge Base Compilation
The script `generate_kb.py` translates the JSON datasets into normalized first-order Prolog relations (`knowledge_base.pl`) declaring `track/8` and `user/7` predicates.

In [2]:
# Execute the ETL compiler to refresh knowledge_base.pl
import generate_kb
generate_kb.main()

# Inspect the first 16 lines of emitted Prolog facts
with open('knowledge_base.pl', 'r', encoding='utf-8') as f:
    lines = [next(f) for _ in range(16)]
print(''.join(lines))


[*] Loading raw JSON datasets...
[*] Emitting Prolog facts to knowledge_base.pl...
[+] Knowledge base compiled successfully: 30 tracks and 6 user profiles written to knowledge_base.pl.

% =========================================================================
% KNOWLEDGE BASE PROLOG: Decision Support System (DSS)
% Automatically generated by generate_kb.py from Spotify & Smartwatch data
% =========================================================================

% --- Facts: track(Id, Title, Artist, Genre, BPM, Valence, Energy, Instrumentalness) ---
:- dynamic track/8.

track(weightless_ambient, 'Weightless Ambient Drift', 'NeuroHarmonics', ambient, 60.0, 0.35, 0.12, 0.96).
track(clair_de_lune, 'Clair de Lune (Debussy)', 'Sinfonia Chambers', classical, 64.0, 0.42, 0.18, 0.94).
track(delta_sleep_stream, 'Delta Wave Binaural Serenade', 'SomnoLab', ambient, 55.0, 0.25, 0.08, 0.99).
track(gymnopedie_no1, 'GymnopǸdie No. 1 (Satie)', 'Aura Piano Quartet', classical, 66.0, 0.38, 0.15, 0.95)

## 3. Inference Engine: Hierarchical AND-OR Goal Tree (`dss_rules.pl`)
The reasoning architecture is structured as a multi-level deductive tree:

- **Level 0 (Meta-Rule):** `recommendable_track(Track, User)` enforces the simultaneous satisfaction of 4 orthogonal evaluation pillars:
  1. `physiologically_compatible/2`: Synchronizes autonomic nervous arousal (sympathetic vs parasympathetic) with musical tempo and energy.
  2. `cognitively_compatible/2`: Shields the phonological loop from lyrical intrusion during intellectual focus (`instrumentalness >= 0.70`).
  3. `safety_and_contraindications/2`: Negative safety barrier against acute stress tachycardia and exhaustion.
  4. `circadian_coherent/2`: Harmonizes audio intensity with the biological sleep/wake cycle.
- **Level 2 & 3 (Sub-rules & Primitives):** Categorize qualitative clinical states before computing the multi-attribute utility score.

In [3]:
from test_suite import load_data, recommend

tracks_map, users_map = load_data()

scenarios = [
    ('user_sarah', 'CASE 1: Acute Work Stress & Autonomic Regulation'),
    ('user_alex',  'CASE 2: High-Intensity Aerobic Cadence Lock'),
    ('user_marco', 'CASE 3: Lyric-Free Deep Focus Study Session')
]

for uid, title in scenarios:
    u = users_map[uid]
    best_track, score = recommend(uid, tracks_map, users_map)
    print('=' * 72)
    print(f' {title}')
    print('=' * 72)
    print(f' User: {u["name"]} | Activity: {u["current_activity"]}')
    print(f' Telemetry: Heart Rate: {u["heart_rate_bpm"]} BPM | Stress: {u["stress_level"]}/100')
    print(f'\n -> RECOMMENDED TRACK: \'{best_track["title"]}\' by {best_track["artist"]}')
    print(f' -> Multi-Attribute Utility: {score:.1f}/100 | Genre: {best_track["genre"]}')
    print(f' -> Audio Attributes: {best_track["bpm"]} BPM, Energy: {best_track["energy"]}, Instrumentalness: {best_track["instrumentalness"]}')
    print()


 CASE 1: Acute Work Stress & Autonomic Regulation
 User: Sarah Miller | Activity: high_stress_work
 Telemetry: Heart Rate: 114.0 BPM | Stress: 84.0/100

 -> RECOMMENDED TRACK: 'Peaceful Mind Breeze' by Zenith Soundscapes
 -> Multi-Attribute Utility: 90.9/100 | Genre: ambient
 -> Audio Attributes: 68.0 BPM, Energy: 0.22, Instrumentalness: 0.91

 CASE 2: High-Intensity Aerobic Cadence Lock
 User: Alex Chen | Activity: cardio_running
 Telemetry: Heart Rate: 146.0 BPM | Stress: 45.0/100

 -> RECOMMENDED TRACK: 'Subterranean Stamina Loop' by Klangwerk
 -> Multi-Attribute Utility: 88.7/100 | Genre: techno
 -> Audio Attributes: 140.0 BPM, Energy: 0.92, Instrumentalness: 0.9

 CASE 3: Lyric-Free Deep Focus Study Session
 User: Marco Rossi | Activity: deep_focus_study
 Telemetry: Heart Rate: 74.0 BPM | Stress: 32.0/100

 -> RECOMMENDED TRACK: 'Raindrops on University Library' by Coffeehouse Chill
 -> Multi-Attribute Utility: 90.8/100 | Genre: lofi
 -> Audio Attributes: 76.0 BPM, Energy: 0.34, I

## 4. Altshuler's Iso-Principle (Clinical Music Therapy)
When a subject experiences high physiological arousal (e.g. 114 BPM during stress), abruptly playing an ultra-slow song induces affective rejection and cognitive dissonance.

**Altshuler's Iso-Principle** solves this using an adaptive 3-stage entrainment curve:
- **Stage 1 (Entry / Pacing):** Meets the user at their elevated tempo (105 BPM).
- **Stage 2 (Bridge / Taper):** Initiates gradual autonomic deceleration (85 BPM).
- **Stage 3 (Target / Lock):** Guides heart rate toward parasympathetic rest (60 BPM).

In [4]:
from test_suite import test_iso_principle

t1, t2, t3 = test_iso_principle(tracks_map, users_map)

print('+======================================================================+')
print('| MULTI-TRACK ENTRAINMENT SESSION (ALTSHULER ISO-PRINCIPLE)            |')
print('+======================================================================+')
print(f'| 1. [ENTRY]     {t1["title"]:28} ({t1["bpm"]:.0f} BPM, E:{t1["energy"]:.2f})             |')
print(f'| 2. [BRIDGE]    {t2["title"]:28} ({t2["bpm"]:.0f} BPM, E:{t2["energy"]:.2f})             |')
print(f'| 3. [TARGET]    {t3["title"]:28} ({t3["bpm"]:.0f} BPM, E:{t3["energy"]:.2f})             |')
print('+----------------------------------------------------------------------+')
print(f'| Trajectory: {t1["bpm"]:.0f} BPM -> {t2["bpm"]:.0f} BPM -> {t3["bpm"]:.0f} BPM (Controlled Downregulation)  |')
print('+======================================================================+')


+======================================================================+
| MULTI-TRACK ENTRAINMENT SESSION (ALTSHULER ISO-PRINCIPLE)            |
+======================================================================+
| 1. [ENTRY]     Iso Bridge Dynamic Transition (105 BPM, E:0.55)             |
| 2. [BRIDGE]    Tempo Taper Andante          (85 BPM, E:0.38)             |
| 3. [TARGET]    Weightless Ambient Drift     (60 BPM, E:0.12)             |
+----------------------------------------------------------------------+
| Trajectory: 105 BPM -> 85 BPM -> 60 BPM (Controlled Downregulation)  |
+======================================================================+


## 5. Dedicated Execution of External Test Script `test_suite.py`
In this concluding section, we directly trigger the external verification test script ([`test_suite.py`](test_suite.py)).

In an interactive Jupyter environment, you can run `%run test_suite.py` or invoke `test_suite.main()`.

The cell below is **fully pre-executed** and displays the complete verification report with all 10 logical properties passing:

In [5]:
# Direct execution of external Python test script 'test_suite.py'
# (In Jupyter you can also run the magic command: %run test_suite.py)
import test_suite
test_suite.main()



 LOGICAL VERIFICATION TEST SUITE (HIERARCHICAL AND-OR GOAL TREE)
 [+] 1. Physiological Sedative Pillar (Sarah) -> Slow, instrumental ambient track: PASSED
 [+] 2. Physiological Ergogenic Pillar (Alex) -> High cadence and maximal energy: PASSED
 [+] 3. Cognitive Protection Pillar (Marco) -> Zero vocal intrusion (instrumental): PASSED
 [+] 4. Circadian Sleep Coherence (David) -> Somnolent acoustic regime (BPM <= 64): PASSED
 [+] 5. Post-Workout Homeostatic Recovery (Elena) -> Moderate controlled activation: PASSED
 [+] 6. Active Commute Vitality (Clara) -> Positive valence and moderate pacing: PASSED
 [+] 7. Iso-Principle Entrainment -> Downward monotonicity (Entry -> Bridge -> Target): PASSED
 [+] 8. Medical Safety Barrier (Negative Test) -> Prohibit high-energy track for Sarah: PASSED
 [+] 9. Cognitive Barrier (Negative Test) -> Prohibit lyrical track for Marco in study: PASSED
 [+] 10. Multi-Attribute Scoring -> Affirmative bonus for user genre affinity: PASSED
----------------------